In [9]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# =========================================================
# 1. PATH SETUP
# =========================================================
PLACEKEY_FLOW_FILE = "../data/intermediate/park_flows_placekey.csv"
PROPERTY_FLOW_FILE = "../data/output/park_flows_property_10km.csv"
AUDIT_FILE         = "../data/output/parks_audit.csv"
ACS_FILE           = "../data/output/census_tracts_acs_2022.csv"

# Census sentinel value for missing/suppressed ACS estimates
ACS_MISSING = -666666666

In [10]:
# =========================================================
# 2. LOAD DATA
# =========================================================
print("Loading data...")
pk_flows   = pd.read_csv(PLACEKEY_FLOW_FILE)
prop_flows = pd.read_csv(PROPERTY_FLOW_FILE)
audit      = pd.read_csv(AUDIT_FILE)
acs        = pd.read_csv(ACS_FILE)

print(f"Placekey flows : {len(pk_flows):,} rows  | cols: {list(pk_flows.columns)}")
print(f"Property flows : {len(prop_flows):,} rows  | cols: {list(prop_flows.columns)}")
print(f"Audit          : {len(audit):,} rows  | cols: {list(audit.columns)}")
print(f"ACS tracts     : {len(acs):,} rows  | cols: {list(acs.columns)}")

Loading data...
Placekey flows : 216,367 rows  | cols: ['tract_i', 'park_j', 'distance_km', 'visits']
Property flows : 100,851 rows  | cols: ['tract_i', 'gis_prop_num', 'property_name', 'visits', 'distance_km', 'all_placekeys']
Audit          : 2,851 rows  | cols: ['placekey', 'parent_placekey', 'name', 'gis_prop_num', 'property_name', 'visits', 'acres', 'forever_wild_id']
ACS tracts     : 2,327 rows  | cols: ['geoid', 'tract_name', 'state', 'county', 'tract', 'total_population', 'median_household_income', 'white_pct', 'black_pct', 'aian_pct', 'asian_pct', 'nhpi_pct', 'other_race_pct', 'two_or_more_races_pct', 'under_15_pct', 'over_65_pct']


In [ ]:
# =========================================================
# 3. CLEAN ACS & BUILD STANDARD 11-DIGIT GEOID
#
# Reconstruct from state / county / tract integer columns
# into Census 11-digit FIPS: state(2) + county(3) + tract(6)
# e.g. '36005000100'
# =========================================================

# We rebuild the GEOID from the integer columns rather than trusting the
# CSV's geoid column — older versions of 01_acs.ipynb wrote it without
# zero-padding, producing 6-digit strings instead of 11. Padding here is
# defensive: it works regardless of how the CSV was generated.
acs['geoid_full'] = (
    acs['state'].astype(str).str.zfill(2) +
    acs['county'].astype(str).str.zfill(3) +
    acs['tract'].astype(str).str.zfill(6)
)

# Census uses -666666666 as a sentinel for suppressed/missing estimates.
# Replace it with NaN across every numeric field so downstream code can
# treat missingness uniformly.
acs_numeric_cols = [
    'total_population', 'median_household_income',
    'white_pct', 'black_pct', 'aian_pct', 'asian_pct',
    'nhpi_pct', 'other_race_pct', 'two_or_more_races_pct',
    'under_15_pct', 'over_65_pct'
]
for col in acs_numeric_cols:
    if col in acs.columns:
        acs[col] = acs[col].replace(ACS_MISSING, np.nan)

print(f"ACS tracts with suppressed median income: {acs['median_household_income'].isna().sum():,}")
print(f"geoid_full sample: {acs['geoid_full'].head(5).tolist()}")

# Subset of ACS columns that get joined onto the flow data later.
# nhpi_pct is excluded — sample sizes are too small in NYC to be useful.
acs_merge_cols = [
    'geoid_full', 'total_population', 'median_household_income',
    'white_pct', 'black_pct', 'aian_pct', 'asian_pct',
    'other_race_pct', 'two_or_more_races_pct',
    'under_15_pct', 'over_65_pct'
]

In [ ]:
# =========================================================
# 3b. IMPUTE MISSING ACS VALUES FROM SPATIAL NEIGHBORS
#
# Some tracts appear in flow data but are absent from the
# ACS pull (e.g. recently split tracts, waterway tracts)
# or have suppressed values (nulled in step 3, e.g. small-
# population tracts with no reported median income).
#
# Method: for each such tract, find all queen-contiguous
# (touching-boundary) neighbors in the NYC tract shapefile
# and replace missing columns with the mean of those
# neighbors' non-null values.  Tracts that are absent from
# the ACS table entirely are added as new rows.
# =========================================================
import geopandas as gpd

TRACT_SHP_PATH = "../data/raw/census/tl_2023_36_tract.shp"
_nyc_counties  = ['005', '047', '061', '081', '085']

# Load the tract polygons in EPSG:2263 so the touches() check uses linear
# (foot-based) geometry. WGS84 lat/lon would be unreliable for adjacency.
tracts_gdf = (
    gpd.read_file(TRACT_SHP_PATH)
    .query("COUNTYFP in @_nyc_counties")
    .to_crs("EPSG:2263")
    .copy()
)
tracts_gdf['geoid_full'] = (
    tracts_gdf['STATEFP'].str.zfill(2) +
    tracts_gdf['COUNTYFP'].str.zfill(3) +
    tracts_gdf['TRACTCE'].str.zfill(6)
)

# Demographic columns that may need imputation
impute_cols = [c for c in acs_merge_cols if c != 'geoid_full']

# Union of tracts referenced by either flow file. If any of these is
# missing ACS data, it would otherwise be dropped by the model.
all_flow_tracts = pd.unique(
    pd.concat([pk_flows['tract_i'], prop_flows['tract_i']]).astype(str)
)

# Index ACS values by geoid for fast neighbor lookup below
acs_vals = acs.set_index('geoid_full')[impute_cols].copy()

# Tracts that we need ACS data for but are missing at least one column
flow_acs   = pd.DataFrame(index=pd.Index(all_flow_tracts, name='geoid_full'))
flow_acs   = flow_acs.join(acs_vals)
missing_tracts = flow_acs[flow_acs.isna().any(axis=1)].index.tolist()

print(f"Unique flow tracts          : {len(all_flow_tracts)}")
print(f"Flow tracts missing any ACS : {len(missing_tracts)}")

# Series of tract polygons keyed by geoid for neighbour searches
geom_lookup = tracts_gdf.set_index('geoid_full')['geometry']

# For each missing tract, find queen-contiguous neighbors and average
# their non-null ACS values. dropna(how='all') means a neighbor with
# all-NaN ACS doesn't pollute the mean.
imputed_records = {}
for tid in missing_tracts:
    if tid not in geom_lookup.index:
        # Tract id appears in flows but not in the shapefile — nothing
        # we can do spatially, so leave it for the dropna in step 5.
        continue
    geom         = geom_lookup[tid]
    neighbor_ids = geom_lookup[geom_lookup.touches(geom)].index
    neighbor_vals = acs_vals.loc[acs_vals.index.isin(neighbor_ids)].dropna(how='all')
    if neighbor_vals.empty:
        continue
    imputed_records[tid] = neighbor_vals.mean()

if imputed_records:
    imputed_df = pd.DataFrame(imputed_records).T
    imputed_df.index.name = 'geoid_full'
    imputed_df = imputed_df.reset_index()

    # Two cases to handle: tracts entirely absent from the ACS get
    # appended as new rows; tracts with partial NaN values get patched
    # in place via update().
    existing_geoids = set(acs['geoid_full'])
    new_rows = imputed_df[~imputed_df['geoid_full'].isin(existing_geoids)].copy()

    if len(new_rows):
        acs = pd.concat([acs, new_rows], ignore_index=True)

    acs = acs.set_index('geoid_full')
    # overwrite=False keeps existing non-null values; only fills NaN cells
    acs.update(imputed_df.set_index('geoid_full'), overwrite=False)
    acs = acs.reset_index()

    print(f"\nTracts imputed from spatial neighbors : {len(imputed_records)}")
    print(f"  New rows added (absent from ACS)    : {len(new_rows)}")
    print(f"  Existing rows patched (suppressed)  : {len(imputed_records) - len(new_rows)}")
    print(f"ACS rows after imputation             : {len(acs)}")
else:
    print("No imputation performed — all flow tracts have complete ACS data.")

In [ ]:
# =========================================================
# 4. BUILD PLACEKEY-LEVEL MODEL DATAFRAME
#
# Merges:
#   (a) park attributes  → join on park_j == placekey    (park side)
#   (b) ACS demographics → join on tract_i == geoid_full (home-tract side)
#   distance_km is already present in pk_flows
#
# Interaction terms are handled in the model formula using
# the var1:var2 syntax, so only raw columns are needed here.
# =========================================================

# (a) Park-side attributes
# A placekey can technically appear under multiple gis_prop_num if its
# point sits exactly on a property boundary. drop_duplicates + groupby
# collapses those duplicates by summing acres (across the few props it
# overlaps) and taking the first FW id.
park_attrs_pk = (
    audit[['placekey', 'gis_prop_num', 'acres', 'forever_wild_id']]
    .drop_duplicates(subset=['placekey', 'gis_prop_num'])
    .groupby('placekey', as_index=False)
    .agg(
        acres           = ('acres', 'sum'),
        forever_wild_id = ('forever_wild_id', 'first')
    )
)

df_pk = pk_flows.merge(park_attrs_pk, left_on='park_j', right_on='placekey', how='left')
print(f"Placekey rows unmatched to audit (acres NaN): {df_pk['acres'].isna().sum():,}")

# (b) Home-tract demographics
# Strip any whitespace from tract IDs to dodge merge mismatches caused by
# stray padding in upstream CSVs.
df_pk['tract_i'] = df_pk['tract_i'].astype(str).str.strip()
df_pk = df_pk.merge(acs[acs_merge_cols], left_on='tract_i', right_on='geoid_full', how='left')
print(f"Placekey rows unmatched to ACS (income NaN): {df_pk['median_household_income'].isna().sum():,}")

# Feature engineering for the gravity-model spec.
# - log_dist / log_area / log_pop: standard log-linear gravity terms.
# - clip(lower=...) avoids -inf when distance, area, or population is 0.
# - is_nature: binary Forever Wild flag.
df_pk['log_dist']         = np.log(df_pk['distance_km'].clip(lower=1e-3))
df_pk['log_area']         = np.log(df_pk['acres'].clip(lower=1e-6))
df_pk['log_pop']          = np.log(df_pk['total_population'].clip(lower=1))
df_pk['is_nature']        = df_pk['forever_wild_id'].notna().astype(int)
df_pk['median_income']    = df_pk['median_household_income']

# Rename pct columns to clean names for readability in model output
df_pk['white']            = df_pk['white_pct']
df_pk['black']            = df_pk['black_pct']
df_pk['asian']            = df_pk['asian_pct']
df_pk['aian']             = df_pk['aian_pct']
df_pk['other_race']       = df_pk['other_race_pct']
df_pk['two_or_more']      = df_pk['two_or_more_races_pct']
df_pk['under_15']         = df_pk['under_15_pct']
df_pk['over_65']          = df_pk['over_65_pct']

# Columns that must be non-NaN for the model to fit
model_cols = [
    'visits', 'is_nature', 'log_dist', 'log_area', 'log_pop',
    'median_income', 'white', 'black', 'asian', 'aian',
    'other_race', 'two_or_more', 'under_15', 'over_65',
    'tract_i'
]

# Drop rows missing any predictor and rows with zero visits (Poisson can't
# distinguish "no flow" from "low flow" in our data — those rows are noise).
df_pk_model = df_pk.dropna(subset=model_cols).copy()
df_pk_model = df_pk_model[df_pk_model['visits'] > 0]

print(f"\nPlacekey model rows (visits > 0, no NaN): {len(df_pk_model):,}")
display(df_pk_model[model_cols[:-1]].describe())

In [ ]:
# =========================================================
# 5. BUILD PROPERTY-LEVEL MODEL DATAFRAME
# =========================================================

# Backfill nature_fraction onto the audit if it was generated by an older
# version of 03_parks.ipynb that didn't include it. Self-contained so this
# notebook runs even against a stale audit CSV.
if 'nature_fraction' not in audit.columns:
    print("nature_fraction missing from audit — computing from Forever Wild CSV...")
    NYC_WILD_CSV_PATH = "../data/raw/forever_wild/NYC_Parks_Forever_Wild_20260205.csv"

    # Load FW Acres, coerce to numeric (the shape-WKT column can break
    # auto-inference), then sum across all FW sites in a property.
    fw_acres = (
        pd.read_csv(NYC_WILD_CSV_PATH)
        .rename(columns=lambda c: c.strip())
        [['GISPropNum', 'Acres']]
        .rename(columns={'GISPropNum': 'gis_prop_num', 'Acres': 'fw_acres'})
    )
    fw_acres['fw_acres'] = pd.to_numeric(fw_acres['fw_acres'], errors='coerce')
    fw_acres = fw_acres.groupby('gis_prop_num', as_index=False)['fw_acres'].sum()

    audit = audit.merge(fw_acres, on='gis_prop_num', how='left')
    audit['fw_acres'] = audit['fw_acres'].fillna(0)
    audit['nature_fraction'] = (audit['fw_acres'] / audit['acres']).clip(lower=0, upper=1)
    # Properties without acreage info can't yield a meaningful fraction
    audit.loc[audit['acres'].isna() | (audit['acres'] <= 0), 'nature_fraction'] = np.nan
    print(f"  nature_fraction computed for {audit['nature_fraction'].notna().sum():,} rows")

# (a) Park-side attributes — collapse audit (one row per placekey) to
# property level. drop_duplicates picks the first row's acres / FW status,
# which are constant across placekeys within a property anyway.
prop_attrs = (
    audit[['gis_prop_num', 'acres', 'forever_wild_id', 'nature_fraction']]
    .drop_duplicates(subset='gis_prop_num')
)

df_prop = prop_flows.merge(prop_attrs, on='gis_prop_num', how='left')
print(f"Property rows unmatched to audit (acres NaN): {df_prop['acres'].isna().sum():,}")

# (b) Home-tract demographics — same merge pattern as the placekey model
df_prop['tract_i'] = df_prop['tract_i'].astype(str).str.strip()
df_prop = df_prop.merge(acs[acs_merge_cols], left_on='tract_i', right_on='geoid_full', how='left')
print(f"Property rows unmatched to ACS (income NaN): {df_prop['median_household_income'].isna().sum():,}")

# Feature engineering. nature_frac is the new continuous flag — fill its
# NaN with 0 so the dropna at the end doesn't kill rows that just have
# missing acreage. is_nature is the original binary FW flag for model #2.
df_prop['log_dist']       = np.log(df_prop['distance_km'].clip(lower=1e-3))
df_prop['log_area']       = np.log(df_prop['acres'].clip(lower=1e-6))
df_prop['log_pop']        = np.log(df_prop['total_population'].clip(lower=1))
df_prop['is_nature']      = df_prop['forever_wild_id'].notna().astype(int)
df_prop['nature_frac']    = df_prop['nature_fraction'].fillna(0).astype(float)
df_prop['median_income']  = df_prop['median_household_income']

df_prop['white']          = df_prop['white_pct']
df_prop['black']          = df_prop['black_pct']
df_prop['asian']          = df_prop['asian_pct']
df_prop['aian']           = df_prop['aian_pct']
df_prop['other_race']     = df_prop['other_race_pct']
df_prop['two_or_more']    = df_prop['two_or_more_races_pct']
df_prop['under_15']       = df_prop['under_15_pct']
df_prop['over_65']        = df_prop['over_65_pct']

model_cols_prop = [
    'visits', 'is_nature', 'nature_frac', 'log_dist', 'log_area', 'log_pop',
    'median_income', 'white', 'black', 'asian', 'aian',
    'other_race', 'two_or_more', 'under_15', 'over_65',
    'tract_i'
]

df_prop_model = df_prop.dropna(subset=model_cols_prop).copy()
df_prop_model = df_prop_model[df_prop_model['visits'] > 0]

print(f"\nProperty model rows (visits > 0, no NaN): {len(df_prop_model):,}")
display(df_prop_model[model_cols_prop[:-1]].describe())

In [ ]:
# =========================================================
# 6. PPML — PLACEKEY LEVEL
# =========================================================

# Poisson Pseudo-Maximum Likelihood (PPML) is the gravity-model standard:
# the log-link Poisson estimator is consistent for E[visits | X] even when
# the count distribution isn't truly Poisson (Santos Silva & Tenreyro 2006).
# `*` in the formula expands to main effects + interaction (var1 + var2 +
# var1:var2), letting each demographic gradient differ by FW status.
formula_pk = """
visits ~ is_nature + log_dist + log_area + log_pop
       + is_nature * median_income
       + is_nature * white
       + is_nature * black
       + is_nature * asian
       + is_nature * aian
       + is_nature * other_race
       + is_nature * two_or_more
       + is_nature * under_15
       + is_nature * over_65
"""

print("Fitting placekey-level PPML...")
res_pk = smf.glm(
    formula=formula_pk,
    data=df_pk_model,
    family=sm.families.Poisson()
).fit(maxiter=200)
print(res_pk.summary())

In [ ]:
# =========================================================
# 7. PPML — PROPERTY LEVEL
# =========================================================

# Same specification as the placekey model but on the property-level data.
# Comparing coefficients between the two models tells us whether the
# demographic gradients hold up after collapsing many POIs into one
# property-level visit count.
formula_prop = """
visits ~ is_nature + log_dist + log_area + log_pop
       + is_nature * median_income
       + is_nature * white
       + is_nature * black
       + is_nature * asian
       + is_nature * aian
       + is_nature * other_race
       + is_nature * two_or_more
       + is_nature * under_15
       + is_nature * over_65
"""

print("Fitting property-level PPML...")
res_prop = smf.glm(
    formula=formula_prop,
    data=df_prop_model,
    family=sm.families.Poisson()
).fit(maxiter=200)
print(res_prop.summary())

In [ ]:
# =========================================================
# 7b. PPML — PROPERTY LEVEL (CONTINUOUS NATURE FRACTION)
#
# Same data as model #2 but replaces the binary is_nature
# flag with nature_frac — the share of property area that
# is Forever Wild, ∈ [0, 1] — both as a main effect and in
# all demographic interaction terms. Lets the "nature
# gradient" scale with how natural the park actually is.
# =========================================================

# A 0 → 1 unit step in nature_frac means "all natural" vs "no natural area
# at all", so coefficients here are NOT directly comparable to model #2 in
# magnitude — only in sign and statistical significance. The interaction
# slope tells us how much the demographic gradient shifts per unit of
# additional natural area.
formula_prop_frac = """
visits ~ nature_frac + log_dist + log_area + log_pop
       + nature_frac * median_income
       + nature_frac * white
       + nature_frac * black
       + nature_frac * asian
       + nature_frac * aian
       + nature_frac * other_race
       + nature_frac * two_or_more
       + nature_frac * under_15
       + nature_frac * over_65
"""

print("Fitting property-level PPML with continuous nature_frac...")
res_prop_frac = smf.glm(
    formula=formula_prop_frac,
    data=df_prop_model,
    family=sm.families.Poisson()
).fit(maxiter=200)
print(res_prop_frac.summary())

In [ ]:
# =========================================================
# 8. COEFFICIENT COMPARISON
# =========================================================

# Demographic predictors that appear both as main effects and as is_nature
# interactions. Used to build the y-axis label list for the plot.
demo_vars = [
    'median_income', 'white', 'black', 'asian',
    'aian', 'other_race', 'two_or_more', 'under_15', 'over_65'
]

# All coefficient names we want to display. The list ordering controls
# the vertical ordering on the chart.
keep_vars = (
    ['is_nature', 'log_dist', 'log_area', 'log_pop'] +
    demo_vars +
    [f'is_nature:{v}' for v in demo_vars]
)

def extract_coefs(res, label):
    # Pull point estimates and 95% CIs from a fitted statsmodels GLM
    # result, restricted to the variables we want to plot. Also tags each
    # row with a model label so we can concat results for plotting.
    available = [v for v in keep_vars if v in res.params.index]
    coefs = res.params[available]
    cis   = res.conf_int().loc[available]
    return pd.DataFrame({
        'variable': available,
        'coef':     coefs.values,
        'ci_lo':    cis[0].values,
        'ci_hi':    cis[1].values,
        'model':    label
    })

# Note: model #3 (continuous nature_frac) is NOT included here because its
# variable names differ (nature_frac instead of is_nature), and a unit
# step has different meaning. Its summary stands alone above.
coef_pk   = extract_coefs(res_pk,   'Placekey')
coef_prop = extract_coefs(res_prop, 'Property')
coef_df   = pd.concat([coef_pk, coef_prop], ignore_index=True)

# Plot — dot for each coefficient, horizontal line for its 95% CI.
# Vertical offsets keep the two models from drawing on top of each other.
plot_vars = coef_pk['variable'].tolist()
fig, ax = plt.subplots(figsize=(12, 10))
colors  = {'Placekey': 'steelblue', 'Property': 'coral'}
offsets = {'Placekey': -0.15, 'Property': 0.15}

for model, grp in coef_df.groupby('model'):
    grp = grp.set_index('variable').reindex(plot_vars).reset_index()
    y = np.arange(len(plot_vars)) + offsets[model]
    ax.scatter(grp['coef'], y, color=colors[model], zorder=3, label=model, s=60)
    for i, (_, row) in enumerate(grp.iterrows()):
        if pd.notna(row['ci_lo']):
            ax.plot([row['ci_lo'], row['ci_hi']], [y[i], y[i]],
                    color=colors[model], alpha=0.6, linewidth=2)

# Dotted divider separates main effects from is_nature interactions
n_main = len(demo_vars) + 4   # is_nature + log_dist + log_area + log_pop + demos
ax.axhline(n_main - 0.5, color='gray', linewidth=0.8, linestyle=':')
ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else -1,
        n_main - 0.35, 'interactions ↑   main effects ↓',
        fontsize=8, color='gray')

# Reference line at zero — coefficients that span it are insignificant
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(np.arange(len(plot_vars)))
ax.set_yticklabels(plot_vars, fontsize=9)
ax.set_xlabel('Coefficient (log-linearized PPML)', fontsize=11)
ax.set_title('PPML Coefficients — Placekey vs Property Level', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

In [ ]:
# =========================================================
# 9. PERSIST MODELS + MODEL FRAMES FOR NOTEBOOK 08
#
# Pickling here lets 08_model_visualization.ipynb skip the
# 10s data-prep + IRLS refit and just load the artefacts.
# =========================================================
import pickle

MODEL_OUTPUT = "../data/intermediate/ppml_models.pkl"

# Bundle everything the visualization notebook needs into one dict.
# Keeping the model dataframes alongside the fitted results means 08 can
# call res.predict(df) without rebuilding any features.
artefacts = {
    'res_pk':         res_pk,
    'res_prop':       res_prop,
    'res_prop_frac':  res_prop_frac,
    'df_pk_model':    df_pk_model,
    'df_prop_model':  df_prop_model,
    'demo_terms':     ['median_income', 'white', 'black', 'asian', 'aian',
                       'other_race', 'two_or_more', 'under_15', 'over_65'],
}

with open(MODEL_OUTPUT, 'wb') as f:
    pickle.dump(artefacts, f)

print(f"Saved fitted models + frames to: {MODEL_OUTPUT}")